In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

from models import SiglipMemeClassifier
from dataset import MemeDataset


def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Hyperparameters
    BATCH_SIZE = 32
    EPOCHS = 10
    LR = 1e-4
    DATA_ROOT = "dataset/split_dataset"

    # Preprocessing (SigLIP expects 224x224)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # SigLIP normalization
    ])

    # Load Data
    train_ds = MemeDataset(DATA_ROOT, 'train', transform=transform)
    val_ds = MemeDataset(DATA_ROOT, 'val', transform=transform)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    # Initialize Model, Loss, Optimizer
    model = SiglipMemeClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    train_losses = []
    validation_losses = []
    # Loop
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            imgs, lbls = imgs.to(device), lbls.to(device)
            
            # validation_loss
            for val_imgs, val_lbls in val_loader:
                val_imgs, val_lbls = val_imgs.to(device), val_lbls.to(device)
                with torch.no_grad():
                    val_logits = model(val_imgs)
                    val_loss = criterion(val_logits, val_lbls)
                    validation_losses.append(val_loss.item())
            
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, lbls)
            
            train_losses.append(loss.item())
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation logic here (omitted for brevity)
        print(f"Epoch {epoch+1} Loss: {train_loss/len(train_loader):.4f}")

    print("Training Complete!")
    return train_losses, validation_losses

def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in dataloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    accuracy = correct / total
    print(f"Validation Accuracy: {accuracy:.4f}")
    return accuracy

/media/storage1/giabao/miniconda3/envs/liver_transplant/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import matplotlib.pyplot as plt

# plotting training and validation losses
def plot_losses(train_losses, validation_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(validation_losses, label='Validation Loss')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show() 

In [3]:
train_losses, validation_losses = train() 
plot_losses(train_losses, validation_losses)

Using device: cuda


Loading weights: 100%|██████████| 208/208 [00:00<00:00, 7756.25it/s]
SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | U

Epoch 1 Loss: 0.6844


Epoch 2: 100%|██████████| 50/50 [02:16<00:00,  2.74s/it]


Epoch 2 Loss: 0.6526


...

Epoch 4 Loss: 0.6041


Epoch 5: 100%|██████████| 50/50 [02:17<00:00,  2.75s/it]


Epoch 5 Loss: 0.5800


Epoch 6:  22%|██▏       | 11/50 [00:32<01:55,  2.95s/it]


KeyboardInterrupt: 